# 05 - PatchTST zero-shot forecasting (summer)

Public research notebook for the STARS workflow. Dataset files and generated artifacts are intentionally excluded from Git. See `../data/README.md` before running.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
(DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "interim").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT / "src"))


In [ ]:
# build_windows_per_building.py
# ----------------------------------------------------------------------
# • One SplitDataset (and DataLoader) for every distinct building_id
# • Same sliding-window logic as before
# ----------------------------------------------------------------------

import pandas as pd, numpy as np, torch
from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader

# ─────────────────────────────────────────────────────────────
# 1.  Sliding-window dataset (unchanged)
# ─────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "electricity_kWh",
    "airTemperature", "relativeHumidity", "windSpeed",
    "month_of_year", "day_of_month", "day_of_week", "hour_of_day",
]

class SplitDataset(Dataset):
    def __init__(self, df, past_len=168, future_len=24, stride=1):
        self.x, self.y = self._build_windows(df, past_len, future_len, stride)

    # --- helpers for date math ---
    @staticmethod
    def _first_monday_july(year: int) -> datetime:
        d = datetime(year, 7, 1)
        while d.weekday() != 0:  # 0 = Monday
            d += timedelta(days=1)
        return d.replace(hour=0, minute=0, second=0, microsecond=0)

    @staticmethod
    def _august_end(year: int) -> datetime:
        return datetime(year, 8, 31, 23, 59, 59)

    # --- core window builder ---
    def _build_windows(self, df, past_len, future_len, stride):
        xs, ys = [], []

        # each df is already a **single** building
        grp = df.sort_values("timestamp").reset_index(drop=True)

        # fill NaNs by nearest neighbour interpolation
        grp_feat = (
            grp[FEATURE_COLS]
            .interpolate(method="nearest", limit_direction="both")
        )

        times = grp["timestamp"].to_numpy()
        data  = grp_feat.to_numpy(np.float32)
        if len(times) == 0:
            return self._empty(past_len, future_len)

        year   = pd.Timestamp(times[0]).year
        anchor = self._first_monday_july(year)
        cutoff = self._august_end(year)

        i = np.searchsorted(times, np.datetime64(anchor), side="left")

        while (
            i + past_len + future_len <= len(data)
            and pd.Timestamp(times[i]).to_pydatetime() <= cutoff
        ):
            past   = data[i : i + past_len]
            future = data[i + past_len : i + past_len + future_len]
            xs.append(past)
            ys.append(future)
            i += stride

        if xs:
            return np.stack(xs), np.stack(ys)
        return self._empty(past_len, future_len)

    @staticmethod
    def _empty(past_len, future_len):
        return (
            np.empty((0, past_len, len(FEATURE_COLS)), np.float32),
            np.empty((0, future_len, len(FEATURE_COLS)), np.float32),
        )

    # torch boiler-plate
    def __len__(self):          return len(self.x)
    def __getitem__(self, idx): return (
        torch.from_numpy(self.x[idx]),
        torch.from_numpy(self.y[idx]),
    )

# ─────────────────────────────────────────────────────────────
# 2.  One DataLoader per building_id
# ─────────────────────────────────────────────────────────────
def make_building_dataloaders(
    data_csv,
    past_len    = 168,
    future_len  = 24,
    stride      = 1,
    batch_size  = 64,
    num_workers = 0,
):
    df = (
        pd.read_csv(data_csv, parse_dates=["timestamp"])
          .sort_values(["building_id", "timestamp"])
    )

    loaders = {}
    for bid, grp in df.groupby("building_id", sort=False):
        ds = SplitDataset(grp, past_len, future_len, stride)
        if len(ds) == 0:
            # optional: print or log empty buildings
            continue

        loaders[bid] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=True,
        )

    return loaders

In [ ]:
train_loaders = make_building_dataloaders(
    data_csv   = str(DATA_DIR / "processed" / "combined_ready_raw.csv"),
    past_len   = 336,
    future_len = 24,
    stride     = 12,
    batch_size = 64,
)

# Peek at one building
sample_bid = next(iter(train_loaders))
x, y = next(iter(train_loaders[sample_bid]))
print(f"Building {sample_bid} ➜ x:", x.shape, " y:", y.shape)
# e.g. (64, 336, 8)  (64, 24, 8)

In [ ]:
# build_windows_per_building_julaug_stride12.py
# ----------------------------------------------------------------------
# • ONE SplitDataset + DataLoader PER building_id
# • From the first Monday 00:00 of July through end of August
# • Sliding windows with configurable stride (default: 12)
# • Prints number of samples per building
# • Interpolate ONLY non-electric features; if electricity has any NaN
#   within a candidate window (past+future), skip that window.
# ----------------------------------------------------------------------

import pandas as pd
import numpy as np
import torch
from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader

FEATURE_COLS = [
    "electricity_kWh",
    "airTemperature", "relativeHumidity", "windSpeed",
    "month_of_year", "day_of_month", "day_of_week", "hour_of_day",
]
AUX_COLS = [
    "airTemperature", "relativeHumidity", "windSpeed",
    "month_of_year", "day_of_month", "day_of_week", "hour_of_day",
]
ELEC_COL = "electricity_kWh"

class SplitDataset(Dataset):
    """
    Multiple (x, y) pairs for a single building, limited to:
      - start: first Monday 00:00 in July (of the first year present)
      - end:   last second of August (same year)
    Shapes:
      x : [N, past_len,   n_features]
      y : [N, future_len, n_features]
    """
    def __init__(self, df, past_len=168, future_len=24, stride=12, interp="time"):
        self.x, self.y = self._build_windows(df, past_len, future_len, stride, interp)

    # ─ helpers (numpy datetime64 for speed/safety) ──────────
    @staticmethod
    def _first_monday_july_np(year: int) -> np.datetime64:
        d = datetime(year, 7, 1)
        while d.weekday() != 0:  # 0 = Monday
            d += timedelta(days=1)
        d = d.replace(hour=0, minute=0, second=0, microsecond=0)
        return np.datetime64(pd.Timestamp(d), "ns")

    @staticmethod
    def _august_end_np(year: int) -> np.datetime64:
        return np.datetime64(pd.Timestamp(year=year, month=8, day=31, hour=23, minute=59, second=59), "ns")

    def _build_windows(self, df, past_len, future_len, stride, interp):
        grp = df.sort_values("timestamp").reset_index(drop=True)
        if grp.empty:
            return self._empty(past_len, future_len)

        # Timestamps
        times = grp["timestamp"].values.astype("datetime64[ns]")

        # 1) Keep electricity_kWh RAW (no interpolation)
        elec = grp[ELEC_COL].to_numpy(dtype=np.float32, copy=False)

        # 2) Interpolate ONLY aux features
        aux = grp[AUX_COLS].copy()
        aux.index = pd.DatetimeIndex(grp["timestamp"].values.astype("datetime64[ns]"))
        try:
            aux = aux.interpolate(method=interp, limit_direction="both")
        except Exception:
            aux = aux.interpolate(method="linear", limit_direction="both")
        aux_np = aux.to_numpy(dtype=np.float32, copy=False)

        # 3) Combine back to full feature matrix (electricity first column)
        data = np.concatenate([elec.reshape(-1, 1), aux_np], axis=1)  # shape (T, 8)

        # Use the year of the first timestamp present
        year = pd.DatetimeIndex(times).year[0]
        anchor = self._first_monday_july_np(year)
        cutoff = self._august_end_np(year)

        # Leftmost index at/after anchor
        i = int(np.searchsorted(times, anchor, side="left"))

        xs, ys = [], []
        # Slide windows entirely within [anchor, cutoff]
        while i + past_len + future_len <= len(data) and times[i] <= cutoff:
            block = data[i : i + past_len + future_len]

            # a) If electricity has ANY NaN in this window, skip
            elec_block = block[:, 0]
            if np.isnan(elec_block).any():
                i += stride
                continue

            # b) If aux still has NaNs after interpolation, also skip (safety)
            if np.isnan(block[:, 1:]).any():
                i += stride
                continue

            past   = block[:past_len]
            future = block[past_len : past_len + future_len]
            xs.append(past)
            ys.append(future)
            i += stride

        if xs:
            return np.stack(xs, dtype=np.float32), np.stack(ys, dtype=np.float32)
        return self._empty(past_len, future_len)

    @staticmethod
    def _empty(past_len, future_len):
        return (
            np.empty((0, past_len, len(FEATURE_COLS)), np.float32),
            np.empty((0, future_len, len(FEATURE_COLS)), np.float32),
        )

    # torch boilerplate
    def __len__(self):          return len(self.x)
    def __getitem__(self, idx): return (
        torch.from_numpy(self.x[idx]),
        torch.from_numpy(self.y[idx]),
    )

# ----------------------------------------------------------------------
# 2.  One DataLoader per building_id + sample counts
# ----------------------------------------------------------------------
def make_building_dataloaders(
    data_csv,
    past_len    = 168,
    future_len  = 24,
    stride      = 12,
    batch_size  = 64,
    num_workers = 0,
    interp      = "time",
    pin_memory  = None,
    drop_last   = False,
):
    df = (
        pd.read_csv(data_csv, parse_dates=["timestamp"])
          .sort_values(["building_id", "timestamp"])
    )

    if pin_memory is None:
        pin_memory = torch.cuda.is_available()

    loaders = {}
    sample_counts = {}

    for bid, grp in df.groupby("building_id", sort=False):
        ds = SplitDataset(grp, past_len=past_len, future_len=future_len, stride=stride, interp=interp)
        n = len(ds)
        if n == 0:
            continue

        loaders[bid] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=drop_last,
        )
        sample_counts[bid] = n

    # Print counts (sorted by building_id for readability)
    for bid in sorted(sample_counts):
        print(f"Building {bid}: {sample_counts[bid]} samples")

    return loaders, sample_counts

# ----------------------------------------------------------------------
# 3.  Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    loaders, counts = make_building_dataloaders(
        data_csv   = str(DATA_DIR / "processed" / "bdg2_ny_2017_with_weather.csv"),
        past_len   = 336,
        future_len = 24,
        stride     = 12,
        batch_size = 64,
    )

    # Peek at one building (if any)
    if loaders:
        sample_bid = next(iter(loaders))
        xb, yb = next(iter(loaders[sample_bid]))
        print(f"Example ➜ Building {sample_bid}  x {xb.shape}  y {yb.shape}")
        # e.g. x: (64, 336, 8)  y: (64, 24, 8)
    else:
        print("No buildings with valid July–August windows were found.")


In [ ]:
from models.PatchTST import Model          # import the class you pasted

# 1) Configuration object -------------------------------------------------
class CFG:
    # core task & data lengths
    task_name = "long_term_forecast"   # or short_term_forecast / imputation …
    seq_len   = 336                    # look-back steps L
    pred_len  = 24                     # forecast horizon  T
    enc_in    = 8                      # total historical channels
    c_out     = 1                      # predict only the 1st channel
    features  = "MS"                   # multi-in, single-out

    # transformer backbone
    d_model   = 32
    d_ff      = 64
    n_heads   = 4
    e_layers  = 3
    dropout   = 0.1
    activation= "gelu"

    # other args required by EncoderLayer signatures
    factor     = 3
    top_k      = 3
    num_kernels= 6

cfg = CFG()
# ── device and model ──────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# ── build one PatchTST per cluster ──────────────────────────────────
model = Model(cfg, patch_len=16, stride=8).to(DEVICE) 

print(model)
print("\nPatchTST Model created. #Params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# train_all_then_test.py  ▸  train on ALL train_loaders, test on each test_loader
# ──────────────────────────────────────────────────────────────────────────────
from torch.utils.data import ConcatDataset, DataLoader
from tqdm import tqdm
import torch, torch.nn as nn, math, pandas as pd

# ─────────────────────────────────────────────────────────────
# 0)  Hyper-params
# ─────────────────────────────────────────────────────────────
EPOCHS  = 5
LR      = 1e-3
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH   = next(iter(train_loaders.values())).batch_size
N_WORK  = next(iter(train_loaders.values())).num_workers
PINMEM  = next(iter(train_loaders.values())).pin_memory

# ─────────────────────────────────────────────────────────────
# 1)  Build *one* global training DataLoader
# ─────────────────────────────────────────────────────────────
train_concat_ds = ConcatDataset([ldr.dataset for ldr in train_loaders.values()])

train_dl = DataLoader(
    train_concat_ds,
    batch_size  = BATCH,
    shuffle     = True,
    num_workers = N_WORK,
    pin_memory  = PINMEM,
)
print(f"Global train-set: {len(train_concat_ds):,} windows  "
      f"({len(train_loaders)} buildings)")

# ─────────────────────────────────────────────────────────────
# 2)  Metric helpers (unchanged)
# ─────────────────────────────────────────────────────────────
def _accum(out, y):
    err   = out - y
    return (
        torch.sum(err ** 2).item(),          # sse
        torch.sum(err).item(),               # bias
        torch.sum(y).item(),                 # sum_y
        torch.sum(torch.abs(err)).item(),    # sae
        y.numel(),                           # n
    )

def _final(sse, bias, sum_y, sae, n):
    rmse   = math.sqrt(sse / (n - 1))
    mean_y = sum_y / n
    return (
        100.0 * rmse / mean_y,               # CVRMSE %
        100.0 * bias / ((n - 1) * mean_y),   # NMBE %
        sae / n,                             # MAE
    )

# ─────────────────────────────────────────────────────────────
# 3)  Instantiate & train once
# ─────────────────────────────────────────────────────────────
model   = Model(cfg, patch_len=16, stride=8).to(DEVICE)
optim   = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_sse = tr_bias = tr_sumy = tr_sae = tr_n = 0
    for xb, yb in tqdm(train_dl, desc=f"[train] e{epoch:02d}", leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred   = model(xb, None, None, None)[..., 0]
        loss   = loss_fn(pred, yb[..., 0])

        optim.zero_grad(); loss.backward(); optim.step()

        se, b, sy, sae, n = _accum(pred, yb[..., 0])
        tr_sse += se; tr_bias += b; tr_sumy += sy; tr_sae += sae; tr_n += n

    cvrm, nmbe, mae = _final(tr_sse, tr_bias, tr_sumy, tr_sae, tr_n)
    print(f"Epoch {epoch:02d}  •  Train CVRMSE {cvrm:6.2f} %  "
          f"NMBE {nmbe:6.2f} %  MAE {mae:8.4f}")

In [ ]:
full_model = model

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4)  Evaluate per test building
# ─────────────────────────────────────────────────────────────
results = []

full_model.eval()
with torch.no_grad():
    for test_bid, test_dl in loaders.items():
        ts_sse = ts_bias = ts_sumy = ts_sae = ts_n = 0
        for xb, yb in test_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred   = full_model(xb, None, None, None)[..., 0]
            se, b, sy, sae, n = _accum(pred, yb[..., 0])
            ts_sse += se; ts_bias += b; ts_sumy += sy; ts_sae += sae; ts_n += n

        cvrm, nmbe, mae = _final(ts_sse, ts_bias, ts_sumy, ts_sae, ts_n)
        results.append({
            "test_building" : test_bid,
            "val_CVRMSE(%)" : cvrm,
            "val_NMBE(%)"   : nmbe,
            "val_MAE"       : mae,
        })
        print(f"✓ {test_bid:>8}  →  CVRMSE {cvrm:6.2f}%  NMBE {nmbe:6.2f}%  MAE {mae:8.4f}")

# ─────────────────────────────────────────────────────────────
# 5)  Save results
# ─────────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
csv_path   = str(RESULTS_DIR / "results_all_train_july_1.csv")
df_results.to_csv(csv_path, index=False)
print(f"\nSaved per-building metrics → {csv_path}")
